# Kenya maize farm risk score: regression models

**Mission.** My mission is to build Africa's capacity to make things, by starting and running technology companies on the continent. This project builds one piece of that: digital infrastructure for farm insurance.

**Problem.** Most smallholder maize farmers in Kenya farm uninsured, so one bad season can wipe them out and lock them out of credit.

**Use case.** The prediction is used by enrolment agents and underwriters at agricultural insurers in Kenya. When a farmer signs up, before planting, the model predicts the field's expected maize yield from what is known at that moment: the field's location and soil, and the inputs the farmer plans to use. The insurer turns the prediction into a low, medium or high risk band to decide cover and price.

**Target.** `yield_kg_ha`: maize grain yield in kilograms per hectare, a continuous number.

**Data.** Aramburu Merlos, F., Tenorio, F. A., Mashingaidze, N., Sananka, A., Aston, S., & Grassini, P. (2024). *Maize management and yield of smallholder farmers in Sub-Saharan Africa between 2016 and 2022* (Version 2) [Dataset]. Zenodo. https://doi.org/10.5281/zenodo.11122388 (CC BY 4.0). One Acre Fund staff measured each field's yield at harvest in two randomly placed 36 m² boxes, and the authors screened the yields for outliers with a Bonferroni outlier test.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
TARGET = "yield_kg_ha"

raw = pd.read_csv("One_Acre_Fund_MEL_maize_survey_data_2016-2022.csv")
print(f"{raw.shape[0]:,} rows, {raw.shape[1]} columns")
raw["country"].value_counts()

14,773 rows, 53 columns


country
kenya       7258
tanzania    3710
rwanda      2080
zambia       861
burundi      640
nigeria      115
uganda       109
Name: count, dtype: int64

## Focus on Kenya

Insurers price risk market by market, so the model covers one country. Kenya has the most fields in the survey, and all of them are from the main (long-rains) season, so every row describes the same kind of decision.

In [2]:
kenya = raw[raw["iso"] == "KEN"].reset_index(drop=True)
print(f"{kenya.shape[0]:,} rows, {kenya.shape[1]} columns")
print("Years:", sorted(kenya["year"].unique().tolist()))
print("Seasons:", list(kenya["season"].unique()))
kenya.head()

7,258 rows, 53 columns
Years: [2016, 2017, 2018, 2019, 2020]
Seasons: ['first season']


,year,country,iso,season,cz,lon,lat,strat,avg_season_gdd,avg_season_ai,avg_season_tavg,season_prec,season_prec_1,season_prec_2,season_prec_3,elev,twi,soil_rzpawhc,soil_clay,soil_pH,soil_orgC,soil_ECEC,plant_date,harvest_date,plant_date_dev,plant_doy,pl_m2,row_spacing,hybrid,hyb_type,hyb_mat,hyb_yor,hyb_tol_mln,hyb_tol_msv,hyb_tol_gls,hyb_tol_nclb,hyb_tol_rust,hyb_tol_ear_rot,N_kg_ha,P_kg_ha,K_kg_ha,compost,comp_kg_ha,manure,fert_in_hole,lime_kg_ha,weeding,pesticide,disease,pest,striga,water_excess,yield_kg_ha
0,2016,kenya,KEN,first season,7701,34.067,0.227,01_KEN7701S1,2509.099854,0.998492,21.275661,677.039708,228.759519,365.227443,83.052746,1280,8.322142,120.0,34.0,6.3,11.182494,13.879732,2016-04-13,NaN,22.0,104.0,NaN,70.666656,True,NaN,2.0,2004.0,False,False,False,False,False,False,51.892050,24.71050,0.0,False,0.0,NaN,True,0.0,NaN,NaN,False,False,NaN,False,1462.951369
1,2017,kenya,KEN,first season,7701,34.100,0.238,01_KEN7701S1,2552.900146,1.038017,21.507408,760.532583,154.624451,464.134379,141.773752,1288,10.117007,120.0,38.0,5.9,12.463738,12.463738,2017-03-31,2017-07-14,8.0,90.0,2.736283,68.666664,True,NaN,2.0,2004.0,False,False,False,False,False,False,51.892050,24.71050,0.0,False,0.0,NaN,True,0.0,False,True,False,True,NaN,False,2817.771269
2,2019,kenya,KEN,first season,7701,34.082,0.247,01_KEN7701S1,2580.099854,1.008526,21.651323,724.153837,181.676820,359.354439,183.122578,1243,7.909871,120.0,33.0,6.2,10.023176,10.023176,2019-04-15,2019-08-05,23.0,105.0,4.541739,72.666664,True,NaN,2.0,2004.0,False,False,False,False,False,False,27.873444,13.83788,0.0,False,0.0,NaN,True,0.0,False,False,False,True,NaN,False,0.000000
3,2020,kenya,KEN,first season,7701,34.098,0.232,01_KEN7701S1,2562.800049,1.018445,21.559790,998.342599,379.451999,428.776234,190.114366,1271,8.969049,120.0,32.0,6.0,10.023176,10.023176,2020-03-28,2020-07-16,6.0,88.0,3.986111,NaN,True,3_way_hyb,1.0,2003.0,False,False,True,False,False,True,44.478900,49.42100,0.0,True,NaN,True,False,0.0,True,False,False,True,True,False,4788.722787
4,2020,kenya,KEN,first season,7701,34.094,0.272,01_KEN7701S1,2612.800049,1.023641,21.824339,998.342599,379.451999,428.776234,190.114366,1226,6.008638,120.0,33.0,5.9,12.463738,11.182494,2020-02-05,2020-08-04,-46.0,36.0,2.277778,NaN,True,NaN,2.0,2004.0,False,False,False,False,False,False,51.892050,24.71050,0.0,False,NaN,NaN,False,0.0,False,NaN,False,False,True,False,2193.982268


In [3]:
features_all = kenya.drop(columns=TARGET)
numeric_columns = features_all.select_dtypes("number").columns
categorical_columns = features_all.columns.difference(numeric_columns)
print(f"{len(kenya):,} rows and {features_all.shape[1]} feature columns: "
      f"{len(numeric_columns)} numeric, {len(categorical_columns)} categorical (text or yes/no)")
print("Categorical columns:", list(categorical_columns))
kenya[TARGET].describe().round(1)

7,258 rows and 52 feature columns: 29 numeric, 23 categorical (text or yes/no)
Categorical columns: ['compost', 'country', 'disease', 'fert_in_hole', 'harvest_date', 'hyb_tol_ear_rot', 'hyb_tol_gls', 'hyb_tol_mln', 'hyb_tol_msv', 'hyb_tol_nclb', 'hyb_tol_rust', 'hyb_type', 'hybrid', 'iso', 'manure', 'pest', 'pesticide', 'plant_date', 'season', 'strat', 'striga', 'water_excess', 'weeding']


count    7258.0
mean     3019.7
std      1707.2
min         0.0
25%      1806.8
50%      2764.6
75%      3973.6
max      9903.5
Name: yield_kg_ha, dtype: float64